# WC_MERCURY_BADGEEVENT_F ETL - ODI to Databricks Migration

**Original Package:** WC_MERCURY_BADGEEVENT_F Load

**Source Schema:** `workspace.PRXBI_TS`

**Target Schema:** `workspace.PRXBI_DW`

---

## Step 1: Define Widgets/Parameters



In [0]:
%sql
-- Create widgets for parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'EOD';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '1';

---

## Step 2: Get ETL Parameters

In [0]:
%sql
-- Get last extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT etl_last_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Get current extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT etl_current_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Get ROW_WID for ETL parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT ROW_WID 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Display ETL parameters
SELECT 
    'Last Extract Time' AS parameter,
    etl_last_extract_time AS value
FROM v_etl_last_extract_time
UNION ALL
SELECT 
    'Current Extract Time' AS parameter,
    etl_current_extract_time AS value
FROM v_etl_current_extract_time
UNION ALL
SELECT 
    'ETL ROW_WID' AS parameter,
    CAST(ROW_WID AS STRING) AS value
FROM v_etl_row_wid;

parameter,value
Last Extract Time,2025-10-20T00:00:00.000Z
Current Extract Time,2026-01-07T00:00:00.000Z
ETL ROW_WID,+21937-01-01T00:00:00.000Z


---

## Step 3: Create Staging Table (C$_0A10DA20FTVLUG38H7LVMMI5D4D)

In [0]:
%sql
-- Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0badgeevent_stg;

In [0]:
%sql
-- Create staging table
CREATE TABLE workspace.PRXBI_DW.c_0badgeevent_stg (
    EVENTEDITIONGBSCODE STRING,
    EVENTTYPE STRING,
    BADGEID STRING,
    SOURCE STRING,
    PRODUCTCODE STRING,
    CUSTOMERTYPE STRING,
    EVENTDATE STRING,
    CREATEDDATE STRING
)
USING DELTA;

---

## Step 4: Extract and Load Data into Staging (with Deduplication)

In [0]:
%sql
-- Extract distinct badge events with row number ranking and time-based filtering
INSERT INTO workspace.PRXBI_DW.c_0badgeevent_stg
SELECT 
    INLINE_VIEW.EVENTEDITIONGBSCODE,
    INLINE_VIEW.EVENTTYPE,
    INLINE_VIEW.BADGEID,
    INLINE_VIEW.SOURCE,
    INLINE_VIEW.PRODUCTCODE,
    INLINE_VIEW.CUSTOMERTYPE,
    INLINE_VIEW.EVENTDATE,
    INLINE_VIEW.CREATEDDATE
FROM (
    SELECT 
        AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,
        AMERCURY_BADGEEVENT_TS.EVENTTYPE,
        AMERCURY_BADGEEVENT_TS.BADGEID,
        AMERCURY_BADGEEVENT_TS.SOURCE,
        AMERCURY_BADGEEVENT_TS.PRODUCTCODE,
        AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE,
        AMERCURY_BADGEEVENT_TS.EVENTDATE,
        AMERCURY_BADGEEVENT_TS.CREATEDDATE,
        ROW_NUMBER() OVER (
            PARTITION BY 
                AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,
                AMERCURY_BADGEEVENT_TS.EVENTTYPE,
                AMERCURY_BADGEEVENT_TS.BADGEID,
                AMERCURY_BADGEEVENT_TS.SOURCE,
                AMERCURY_BADGEEVENT_TS.PRODUCTCODE,
                AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE
            ORDER BY AMERCURY_BADGEEVENT_TS.EVENTDATE DESC
        ) AS RN
    FROM (
        SELECT 
            AMERCURY_BADGEEVENT_TS1.EVENTEDITIONGBSCODE,
            AMERCURY_BADGEEVENT_TS1.EVENTTYPE,
            AMERCURY_BADGEEVENT_TS1.BADGEID,
            AMERCURY_BADGEEVENT_TS1.SOURCE,
            AMERCURY_BADGEEVENT_TS1.PRODUCTCODE,
            AMERCURY_BADGEEVENT_TS1.CUSTOMERTYPE,
            MAX(AMERCURY_BADGEEVENT_TS1.INT_INSERT_DATE) AS INT_INSERT_DATE
        FROM workspace.PRXBI_TS.WC_MERCURY_BADGEEVENT_TS AMERCURY_BADGEEVENT_TS1
        WHERE AMERCURY_BADGEEVENT_TS1.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
          AND AMERCURY_BADGEEVENT_TS1.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
        GROUP BY
            AMERCURY_BADGEEVENT_TS1.EVENTEDITIONGBSCODE,
            AMERCURY_BADGEEVENT_TS1.EVENTTYPE,
            AMERCURY_BADGEEVENT_TS1.BADGEID,
            AMERCURY_BADGEEVENT_TS1.SOURCE,
            AMERCURY_BADGEEVENT_TS1.PRODUCTCODE,
            AMERCURY_BADGEEVENT_TS1.CUSTOMERTYPE
    ) AMERCURY_BADGEEVENT_TS1_1
    INNER JOIN workspace.PRXBI_TS.WC_MERCURY_BADGEEVENT_TS AMERCURY_BADGEEVENT_TS
        ON AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE = AMERCURY_BADGEEVENT_TS1_1.EVENTEDITIONGBSCODE
        AND AMERCURY_BADGEEVENT_TS1_1.EVENTTYPE = AMERCURY_BADGEEVENT_TS.EVENTTYPE
        AND AMERCURY_BADGEEVENT_TS1_1.BADGEID = AMERCURY_BADGEEVENT_TS.BADGEID
        AND AMERCURY_BADGEEVENT_TS1_1.SOURCE = AMERCURY_BADGEEVENT_TS.SOURCE
        AND AMERCURY_BADGEEVENT_TS1_1.PRODUCTCODE = AMERCURY_BADGEEVENT_TS.PRODUCTCODE
        AND AMERCURY_BADGEEVENT_TS1_1.CUSTOMERTYPE = AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE
        AND AMERCURY_BADGEEVENT_TS1_1.INT_INSERT_DATE = AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE
    WHERE AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
      AND AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
    GROUP BY
        AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,
        AMERCURY_BADGEEVENT_TS.EVENTTYPE,
        AMERCURY_BADGEEVENT_TS.BADGEID,
        AMERCURY_BADGEEVENT_TS.SOURCE,
        AMERCURY_BADGEEVENT_TS.PRODUCTCODE,
        AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE,
        AMERCURY_BADGEEVENT_TS.EVENTDATE,
        AMERCURY_BADGEEVENT_TS.CREATEDDATE
) INLINE_VIEW
WHERE INLINE_VIEW.RN = 1;

num_affected_rows,num_inserted_rows
553138,553138


In [0]:
%sql
-- Show staging record count
SELECT COUNT(*) AS staging_record_count 
FROM workspace.PRXBI_DW.c_0badgeevent_stg;

staging_record_count
553138


---

## Step 5: Create Integration/Flow Table (I$_WC_MERCURY_BADGEEVENT_F)

In [0]:
%sql
-- Drop integration table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;

In [0]:
%sql
-- Create integration/flow table
CREATE TABLE workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow (
    ROW_WID BIGINT,
    INTEGRATION_ID STRING,
    ETL_PROC_WID BIGINT,
    EVENTEDITIONGBSCODE STRING,
    EVENTTYPE STRING,
    BADGEID STRING,
    SOURCE STRING,
    PRODUCTCODE STRING,
    CUSTOMERTYPE STRING,
    EVENTDATE STRING,
    CREATEDDATE STRING,
    BADGE_WID BIGINT,
    EVENT_EDITION_WID BIGINT,
    OBU_WID BIGINT,
    PRODUCT_WID BIGINT,
    W_UPDATE_DT TIMESTAMP,
    W_INSERT_DT TIMESTAMP,
    EVENT_WID BIGINT,
    IND_UPDATE STRING
)
USING DELTA;

---

## Step 6: Populate Flow Table with Lookups

In [0]:
%sql
-- Insert into flow table with dimension lookups
-- DETECTION_STRATEGY = NONE (no change detection, always insert/update)
INSERT INTO workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow (
    INTEGRATION_ID,
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    SOURCE,
    PRODUCTCODE,
    CUSTOMERTYPE,
    EVENTDATE,
    CREATEDDATE,
    BADGE_WID,
    EVENT_EDITION_WID,
    OBU_WID,
    PRODUCT_WID,
    EVENT_WID,
    IND_UPDATE
)
SELECT 
    CONCAT_WS('~',
        FILTER2_A_1.EVENTEDITIONGBSCODE,
        FILTER2_A_1.EVENTTYPE,
        FILTER2_A_1.BADGEID,
        FILTER2_A_1.SOURCE,
        FILTER2_A_1.PRODUCTCODE,
        FILTER2_A_1.CUSTOMERTYPE
    ) AS INTEGRATION_ID,
    FILTER2_A_1.EVENTEDITIONGBSCODE,
    FILTER2_A_1.EVENTTYPE,
    FILTER2_A_1.BADGEID,
    FILTER2_A_1.SOURCE,
    FILTER2_A_1.PRODUCTCODE,
    FILTER2_A_1.CUSTOMERTYPE,
    FILTER2_A_1.EVENTDATE,
    FILTER2_A_1.CREATEDDATE,
    COALESCE(WC_BADGE_DETAILS_D.ROW_WID, 0) AS BADGE_WID,
    COALESCE(WC_EVENT_ED_D_1.ROW_WID, 0) AS EVENT_EDITION_WID,
    COALESCE(WC_EVENT_ED_D_1.OBU_WID, 0) AS OBU_WID,
    COALESCE(WC_BADGE_PRODUCT_D_SQ_BADGEP_1.ROW_WID_1, 0) AS PRODUCT_WID,
    COALESCE(WC_EVENT_D.ROW_WID, 0) AS EVENT_WID,
    'I' AS IND_UPDATE
FROM (
    SELECT 
        FILTER2_A.EVENTEDITIONGBSCODE,
        FILTER2_A.EVENTTYPE,
        FILTER2_A.BADGEID,
        FILTER2_A.SOURCE,
        FILTER2_A.PRODUCTCODE,
        FILTER2_A.CUSTOMERTYPE,
        FILTER2_A.EVENTDATE,
        FILTER2_A.CREATEDDATE
    FROM workspace.PRXBI_DW.c_0badgeevent_stg FILTER2_A
) FILTER2_A_1
LEFT OUTER JOIN (
    SELECT 
        WC_EVENT_ED_D.ROW_WID,
        WC_EVENT_ED_D.EVENT_EDITION_CODE,
        WC_EVENT_ED_D.EVENT_ALPHA_CODE,
        WC_EVENT_ED_D.OBU_WID,
        WC_EVENT_ED_D.EVENT_INTEGRATION_ID
    FROM workspace.PRXBI_DW.WC_EVENT_ED_D WC_EVENT_ED_D
) WC_EVENT_ED_D_1
    ON FILTER2_A_1.EVENTEDITIONGBSCODE = CONCAT(
        RPAD(WC_EVENT_ED_D_1.EVENT_ALPHA_CODE, 5, '-'),
        WC_EVENT_ED_D_1.EVENT_EDITION_CODE
    )
LEFT OUTER JOIN workspace.PRXBI_DW.WC_BADGE_DETAILS_D WC_BADGE_DETAILS_D
    ON FILTER2_A_1.BADGEID = WC_BADGE_DETAILS_D.BADGE_ID
LEFT OUTER JOIN (
    SELECT 
        MAX(WC_BADGE_PRODUCT_D_SQ_BADGEPRO.ROW_WID) AS ROW_WID,
        WC_BADGE_PRODUCT_D_SQ_BADGEPRO.SKU,
        MAX(WC_BADGE_PRODUCT_D_SQ_BADGEPRO.ROW_WID) AS ROW_WID_1,
        WC_BADGE_PRODUCT_D_SQ_BADGEPRO.SKU AS SKU_1
    FROM workspace.PRXBI_DW.WC_BADGE_PRODUCT_D WC_BADGE_PRODUCT_D_SQ_BADGEPRO
    GROUP BY WC_BADGE_PRODUCT_D_SQ_BADGEPRO.SKU
) WC_BADGE_PRODUCT_D_SQ_BADGEP_1
    ON WC_BADGE_PRODUCT_D_SQ_BADGEP_1.SKU_1 = FILTER2_A_1.PRODUCTCODE
LEFT OUTER JOIN workspace.PRXBI_DW.WC_EVENT_D WC_EVENT_D
    ON WC_EVENT_ED_D_1.EVENT_INTEGRATION_ID = WC_EVENT_D.INTEGRATION_ID
WHERE 1=1;

num_affected_rows,num_inserted_rows
553138,553138


In [0]:
%sql
-- Show flow table record count
SELECT COUNT(*) AS flow_record_count 
FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;

flow_record_count
553138


---

## Step 7: Check for Primary Key Violations (Quality Check)

In [0]:
%sql
-- Check for duplicate INTEGRATION_IDs (PK violations)
SELECT 
    INTEGRATION_ID,
    COUNT(*) AS duplicate_count
FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
GROUP BY INTEGRATION_ID
HAVING COUNT(*) > 1;

INTEGRATION_ID,duplicate_count


In [0]:
%sql
drop table if exists workspace.PRXBI_DW.deduped_flow;
-- Remove duplicates if any exist (keep first occurrence)
CREATE TABLE workspace.PRXBI_DW.deduped_flow
USING DELTA
AS
SELECT *
FROM (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY INTEGRATION_ID ORDER BY EVENTDATE DESC) AS rn
    FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
)
WHERE rn = 1;

-- -- Clear and reload flow table with deduplicated data
DELETE FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;


INSERT INTO workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
SELECT 
    ROW_WID,
    INTEGRATION_ID,
    ETL_PROC_WID,
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    SOURCE,
    PRODUCTCODE,
    CUSTOMERTYPE,
    EVENTDATE,
    CREATEDDATE,
    BADGE_WID,
    EVENT_EDITION_WID,
    OBU_WID,
    PRODUCT_WID,
    W_UPDATE_DT,
    W_INSERT_DT,
    EVENT_WID,
    IND_UPDATE
FROM workspace.prxbi_dw.deduped_flow;

num_affected_rows,num_inserted_rows


---

## Step 8: Mark Records for Update vs Insert

In [0]:
%sql
-- DETECTION_STRATEGY = NONE
-- Mark records that already exist in target as UPDATE
UPDATE workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow F
SET IND_UPDATE = 'U'
WHERE EXISTS (
    SELECT 1
    FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F T
    WHERE T.INTEGRATION_ID = F.INTEGRATION_ID
);

num_affected_rows
553138


In [0]:
%sql
select count(*) from workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
where IND_UPDATE = 'I';

count(*)
0


In [0]:
%sql
-- Show update vs insert breakdown
SELECT 
    IND_UPDATE,
    COUNT(*) AS record_count
FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
GROUP BY IND_UPDATE;

IND_UPDATE,record_count
U,553138


---

## Step 9: Perform MERGE Operation (Update + Insert)

In [0]:
%sql
-- MERGE into target table using Databricks MERGE syntax
MERGE INTO workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F AS T
USING workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.EVENTEDITIONGBSCODE = S.EVENTEDITIONGBSCODE,
    T.EVENTTYPE = S.EVENTTYPE,
    T.BADGEID = S.BADGEID,
    T.SOURCE = S.SOURCE,
    T.PRODUCTCODE = S.PRODUCTCODE,
    T.CUSTOMERTYPE = S.CUSTOMERTYPE,
    T.EVENTDATE = S.EVENTDATE,
    T.CREATEDDATE = S.CREATEDDATE,
    T.BADGE_WID = S.BADGE_WID,
    T.EVENT_EDITION_WID = S.EVENT_EDITION_WID,
    T.OBU_WID = S.OBU_WID,
    T.PRODUCT_WID = S.PRODUCT_WID,
    T.EVENT_WID = S.EVENT_WID,
    T.ETL_PROC_WID = ${ETL_PROC_WID},
    T.W_UPDATE_DT = CURRENT_TIMESTAMP()
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    INTEGRATION_ID,
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    SOURCE,
    PRODUCTCODE,
    CUSTOMERTYPE,
    EVENTDATE,
    CREATEDDATE,
    BADGE_WID,
    EVENT_EDITION_WID,
    OBU_WID,
    PRODUCT_WID,
    EVENT_WID,
    ETL_PROC_WID,
    W_UPDATE_DT,
    W_INSERT_DT
) VALUES (
    S.INTEGRATION_ID,
    S.EVENTEDITIONGBSCODE,
    S.EVENTTYPE,
    S.BADGEID,
    S.SOURCE,
    S.PRODUCTCODE,
    S.CUSTOMERTYPE,
    S.EVENTDATE,
    S.CREATEDDATE,
    S.BADGE_WID,
    S.EVENT_EDITION_WID,
    S.OBU_WID,
    S.PRODUCT_WID,
    S.EVENT_WID,
    ${ETL_PROC_WID},
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
553138,553138,0,0


---

## Step 10: Optimize Target Table

In [0]:
%sql
-- Optimize Delta table for better query performance
-- This is Databricks equivalent of Oracle's dbms_stats.gather_table_stats
OPTIMIZE workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1767881351764, 1767881353229, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, null)"


In [0]:
%sql
-- Optional: Z-order by frequently filtered columns
OPTIMIZE workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F
ZORDER BY (INTEGRATION_ID, EVENT_EDITION_WID, BADGE_WID);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 13130806), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1767881355225, 1767881356639, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, null)"


---

## Step 11: Cleanup - Drop Temporary Tables

In [0]:
%sql
-- Drop integration/flow table
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;

In [0]:
%sql
-- Drop staging table
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0badgeevent_stg;

DROP TABLE IF EXISTS workspace.PRXBI_DW.deduped_flow;

---

## Step 12: Validation & Summary

In [0]:
%sql
-- Final validation - show summary statistics
SELECT 
    'WC_MERCURY_BADGEEVENT_F' AS table_name,
    COUNT(*) AS total_records,
    COUNT(DISTINCT INTEGRATION_ID) AS unique_integration_ids,
    MAX(W_UPDATE_DT) AS last_update_time,
    'ETL Completed Successfully' AS status
FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F;

table_name,total_records,unique_integration_ids,last_update_time,status
WC_MERCURY_BADGEEVENT_F,553138,553138,2026-01-09T09:28:02.824Z,ETL Completed Successfully


In [0]:
%sql
-- Show sample of recently updated records
SELECT 
    INTEGRATION_ID,
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    PRODUCTCODE,
    BADGE_WID,
    EVENT_EDITION_WID,
    W_UPDATE_DT,
    ETL_PROC_WID
FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F
ORDER BY W_UPDATE_DT DESC
LIMIT 10;

INTEGRATION_ID,EVENTEDITIONGBSCODE,EVENTTYPE,BADGEID,PRODUCTCODE,BADGE_WID,EVENT_EDITION_WID,W_UPDATE_DT,ETL_PROC_WID
ADHRW23~BadgeDownloaded~0843488195700411-OWY~EventPortal~PACKAGE-ADHRW-38100003-5~tradeVisitor,ADHRW23,BadgeDownloaded,0843488195700411-OWY,PACKAGE-ADHRW-38100003-5,0,13216,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0843433808530026-1SN~EventPortal~PACKAGE-ADHRW-38100003-2~tradeVisitor,ADHRW23,BadgeDownloaded,0843433808530026-1SN,PACKAGE-ADHRW-38100003-2,0,13216,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0843435819481675-HRV~EventPortal~PACKAGE-ADHRW-38100003-9~tradeVisitor,ADHRW23,BadgeDownloaded,0843435819481675-HRV,PACKAGE-ADHRW-38100003-9,0,13216,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0843452182219591-1LA~EventPortal~PACKAGE-ADHRW-38100003-1~tradeVisitor,ADHRW23,BadgeDownloaded,0843452182219591-1LA,PACKAGE-ADHRW-38100003-1,0,13216,2026-01-08T14:09:06.201Z,1
ADHRN25~BadgeDownloaded~1376703501340131-KKP~EventPortal~PACKAGE-ADHRN-38100022~tradeVisitor,ADHRN25,BadgeDownloaded,1376703501340131-KKP,PACKAGE-ADHRN-38100022,0,16506,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0843433224676636-KKM~EventPortal~PACKAGE-ADHRW-38100004-1~vipOrBuyersProg,ADHRW23,BadgeDownloaded,0843433224676636-KKM,PACKAGE-ADHRW-38100004-1,0,13216,2026-01-08T14:09:06.201Z,1
ADHRN25~BadgeDownloaded~1339488106909098-UBM~EventPortal~PACKAGE-ADHRN-38100022~tradeVisitor,ADHRN25,BadgeDownloaded,1339488106909098-UBM,PACKAGE-ADHRN-38100022,0,16506,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0843423053270358-U2L~EventPortal~PACKAGE-ADHRW-38100003-1~tradeVisitor,ADHRW23,BadgeDownloaded,0843423053270358-U2L,PACKAGE-ADHRW-38100003-1,0,13216,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0840986317591893-22P~EventPortal~PACKAGE-ADHRW-38100009~exhibitingDelegate,ADHRW23,BadgeDownloaded,0840986317591893-22P,PACKAGE-ADHRW-38100009,0,13216,2026-01-08T14:09:06.201Z,1
ADHRW23~BadgeDownloaded~0843494775869754-C32~EventPortal~PACKAGE-ADHRW-38100003-5~tradeVisitor,ADHRW23,BadgeDownloaded,0843494775869754-C32,PACKAGE-ADHRW-38100003-5,0,13216,2026-01-08T14:09:06.201Z,1


---

## Notes on Conversion

### Key Changes from ODI to Databricks:

1. **Sequences Removed**: `WC_MERCURY_BADGEEVENT_F_SEQ.NEXTVAL` removed. If ROW_WID is needed, use identity columns in table definition.

2. **Oracle Hints Removed**: `/*+ append */` hints are not needed in Spark SQL.

3. **SYSTIMESTAMP → CURRENT_TIMESTAMP()**: Standard Spark SQL function.

4. **NOLOGGING Removed**: Not applicable in Delta Lake.

5. **Index Creation Skipped**: Delta Lake handles indexing automatically. OPTIMIZE and Z-ORDER commands replace traditional indexing.

6. **dbms_stats Replaced**: `OPTIMIZE` command handles statistics and compaction.

7. **Error Tables Removed**: E$ tables and SNP_CHECK_TAB replaced with PK validation check in Step 7.

8. **NVL → COALESCE**: Standard SQL function.

9. **String Concatenation**: Oracle's `||` operator replaced with `CONCAT_WS()` for cleaner code.

10. **DETECTION_STRATEGY = NONE**: This means the process always attempts UPDATE if exists, INSERT if not, without comparing values.

11. **Schema References**: 
    - `PRXBI_DW_SEP` → `workspace.PRXBI_DW`
    - `PRXBI_TS_SEP` → `workspace.PRXBI_TS`

### Manual Actions Required:

1. **ROW_WID Column**: If ROW_WID needs to be sequential and persisted, add identity column to target table definition:

2. **Target Table Creation**: If `WC_MERCURY_BADGEEVENT_F` doesn't exist, create it with proper schema before running this notebook.

3. **Dimension Tables**: Ensure these dimension tables exist:
   - `WC_EVENT_ED_D`
   - `WC_BADGE_DETAILS_D`
   - `WC_BADGE_PRODUCT_D`
   - `WC_EVENT_D`

4. **Time-Based Filtering**: The staging extract uses time-based incremental load based on ETL parameters.

---

## Remove Widgets (Optional - run at end)

In [0]:
%sql
-- Remove widgets at the end of the job
-- REMOVE WIDGET ETL_JOB_TYPE;
-- REMOVE WIDGET DATASOURCE_NUM_ID;
-- REMOVE WIDGET ETL_PROC_WID;